# 📐 Modelo TCO Sostenible (Escenario MITMA 2026 - 44 Toneladas)

Este análisis desglosa la estructura de costes de explotación para un vehículo articulado de gran tonelaje, cumpliendo con la metodología del **Observatorio de Costes del Ministerio de Transportes**.

La tarifa técnica se calcula mediante la suma de costes fijos (amortizados por km) y costes variables directos:

$$ \text{Tarifa Técnica} (€/km) = \frac{\sum \text{Costes Fijos Anuales}}{\text{Kilometraje Anual}} + \sum \text{Costes Variables} (€/km) $$

In [46]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

# =====================================================================
# LÍNEA BASE SOSTENIBLE (REFERENCIAS OFICIALES 2026)
# =====================================================================

fixed_costs_base = {
    "personal_y_dietas": 62500.0,   # Coste Total Cargado (Salario + Seg. Soc. + Dietas)
    "amortizacion_vehiculo": 19500.0, # Amortización acelerada por mayor desgaste 44t
    "seguros_y_visados": 4500.0,    # Seguro RC + Mercancía
    "costes_indirectos": 14500.0,   # Gestión estructural avanzada
    "fiscalidad_y_otros": 2000.0     # IVTM y tasas técnicas
}

variable_costs_base = {
    "combustible_diesel": 0.460,    # Consumo real 44t (38L/100km) neto profesional
    "mantenimiento_y_tires": 0.165, # Correctivo + Neumáticos 44t (mayor desgaste)
    "adblue_y_aditivos": 0.015
}

KM_ANUALES_REF = 110000

from logistic_core.utils.cost_estimator import CostEstimator
estimator = CostEstimator(
    fixed_costs_annual=fixed_costs_base,
    variable_costs_km=variable_costs_base,
    annual_km_per_truck=KM_ANUALES_REF
)
print(f"Tarifa Técnica Sostenible (MITMA 2026): {estimator.price_per_km:.4f} €/km")

Tarifa Técnica Sostenible (MITMA 2026): 1.5764 €/km


## 🎯 Escenario Estratégico Objetivo (1.50 €/km)

Buscamos la combinación **mínimamente desviada** de la realidad del mercado (Línea Base MITMA) que justifique una tarifa técnica de **1.5 €/km**.

Utilizaremos un optimizador para ajustar levemente las variables operativas (Km, Diésel, Personal, Amortización e Indirectos).

In [47]:
import numpy as np
from scipy.optimize import minimize

TARGET_RATE = 1.50

def objective(x):
    # x[0]=km, x[1]=fuel, x[2]=salario, x[3]=amort, x[4]=ind
    total_fixed = x[2] + x[3] + x[4] + 4500.0 + 2000.0
    total_variable = x[1] + 0.165 + 0.015
    rate = (total_fixed / x[0]) + total_variable
    return (rate - TARGET_RATE)**2

# RANGOS DE TOLERANCIA ESTRICTOS (Justificables)
bounds = [
    (90000, 120000),  # Kilometraje
    (0.400, 0.550),   # Combustible
    (58000, 68000),   # Salario
    (17000, 23000),   # Amortización
    (12500, 16500)    # Costes Indirectos
]

# Ejecutamos con método SLSQP y alta precisión para clavar el 1.50
res = minimize(objective, [105000, 0.45, 62500, 19500, 14500], bounds=bounds, method='SLSQP', tol=1e-12)

if res.success:
    o_km, o_f, o_s, o_a, o_i = res.x
    print("=== ESCENARIO OPTIMIZADO PARA 1.5 €/km ===")
    print(f"Tarifa Objetivo:         {TARGET_RATE:.2f} €/km")
    print(f"Kilometraje Anual:       {o_km:,.0f} km")
    print(f"Combustible (€/km):      {o_f:.3f}")
    print(f"Coste Personal Anual:    {o_s:,.0f} €")
    print(f"Amortización Vehículo:   {o_a:,.0f} €")
    print(f"Costes Indirectos:       {o_i:,.0f} €")
    
    # Verificación Matemática
    t_fix = o_s + o_a + o_i + 4500.0 + 2000.0
    t_var = o_f + 0.165 + 0.015
    rate_check = (t_fix / o_km) + t_var
    print(f"\nTarifa Resultante Final: {rate_check:.2f} €/km")
else:
    print("No se halló solución exacta. Revise los rangos.")

=== ESCENARIO OPTIMIZADO PARA 1.5 €/km ===
Tarifa Objetivo:         1.50 €/km
Kilometraje Anual:       106,607 km
Combustible (€/km):      0.400
Coste Personal Anual:    60,859 €
Amortización Vehículo:   17,859 €
Costes Indirectos:       12,859 €

Tarifa Resultante Final: 1.50 €/km
